<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Asset_Library_Browser_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Asset Library Browser — browse, tag, preview, export your 200+ game assets

Once you've run TripoSplat + GauStudio / SuGaR + Mesh Optimizer on your 200+ images, you have a folder full of assets but **no way to browse them, pick the best ones, or ship them to a game engine**. This notebook is the missing piece — a Gradio UI that scans your asset library, lets you preview / tag / favorite / filter, render thumbnails, and export to Unity AssetBundle, UE, Godot, or a static HTML portfolio.

**What it does:**

- **Step 1** — install deps (gradio 5.49.1, trimesh, open3d, tqdm, no model weights needed)
- **Step 2** — scan a library folder, recognize formats (3DGS PLY/SPLAT, mesh GLB/OBJ/FBX/PLY, image PNG, FBX, text), build a metadata sidecar JSON (tags, favorites, notes per asset)
- **Step 3** — the main Gradio UI: gallery with thumbnails, filter by tag/format/favorite, click any asset to preview (3D mesh in `<model-viewer>`, 3DGS gets metadata + 'convert to mesh' button), tag editor, favorite toggle
- **Step 4** — batch render thumbnails (renders each mesh to a 256×256 PNG via trimesh's offscreen renderer; 3DGS falls back to file icon or optional turntable GIF via gsplat)
- **Step 5** — export: Unity-style AssetBundle folder structure, Godot .tres, static HTML portfolio (single self-contained file you can host on GitHub Pages), CSV manifest for the library
- **Step 6** — stats dashboard: total assets, format breakdown, top tags, missing-file report, biggest files
- **Step 7** — Drive mirror (whole library + metadata + portfolio HTML)
- **Step 8** — keep-alive (Gradio runs forever otherwise)
- **Step 9** — help / format reference / known issues

**Compute:** CPU-only. The browser is a UI, not a model. No GPU required, no model weights to download. First run: ~3 min install. Subsequent: instant. Can run alongside other notebooks in the same session (read-only on the library folder).

In [ ]:
# @title STEP 1 — Install Dependencies (CPU-only)
# @markdown The Asset Library Browser is a Gradio UI on top of trimesh +
# @markdown open3d + Pillow. No GPU, no model weights, no 3DGS rasterizer
# @markdown needed. ~2 min install.

import os, sys, subprocess, time
t0 = time.time()
print('[browser] Installing lightweight deps ...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'gradio==5.49.1', 'tqdm==4.66.5',
                'trimesh[easy]==4.12.2', 'open3d==0.19.0',
                'scikit-image', 'pillow', 'numpy', 'pandas',
                'pywavefront'], check=True)
print(f'[browser] Deps installed in {time.time()-t0:.1f}s', flush=True)

# Sanity-check
import gradio, trimesh, open3d, PIL, numpy
print(f'[verify] gradio={gradio.__version__} trimesh={trimesh.__version__} '
      f'open3d={open3d.__version__} PIL={PIL.__version__} numpy={numpy.__version__}')
print('\n[Done] STEP 1 complete. Move on to STEP 2 (library scan).')


In [ ]:
# @title STEP 2 — Scan Library + Build Metadata Sidecar
# @markdown Scans a folder of 200+ generated assets, recognizes formats, builds a
# @markdown metadata sidecar JSON with tags / favorites / notes. The sidecar lives
# @markdown in `<LIBRARY_DIR>/_library_meta.json` and is the persistent state.

import os, sys, json, time, pathlib, hashlib
from datetime import datetime, timezone
import numpy as np
from PIL import Image

LIBRARY_DIR = '/content/drive/MyDrive/AEI_3D_Out'  # @param {type:"string"}
# @markdown *Folder containing your generated assets. Usually /content/drive/MyDrive/AEI_3D_Out/. Can be the root or any subfolder (e.g. /triposplat_runs/batch).*
RECURSIVE = True  # @param {type:"boolean"}
# @markdown *Scan subfolders. Recommended ON for the 200+ library (assets are in batch_*/, postprocessed/, etc.)*
INCLUDE_GLOBS = '*'  # @param {type:"string"}
# @markdown *Glob pattern. Default '*' = everything. Set to '*.glb' to only see meshes, '*.ply' for 3DGS + meshes, etc.*
MAX_ASSETS = 0  # @param {type:"integer"}
# @markdown *Cap on the number of assets scanned. 0 = no cap (default). Useful for testing.*
THUMB_DIR_NAME = '_thumbs'  # @param {type:"string"}
# @markdown *Subfolder for rendered thumbnails. Auto-created.*

lib_path = pathlib.Path(LIBRARY_DIR)
if not lib_path.exists():
    raise SystemExit(f'[scan] Library dir {lib_path} not found. Set LIBRARY_DIR to your assets folder.')

META_FILE = lib_path / '_library_meta.json'
THUMB_DIR = lib_path / THUMB_DIR_NAME
THUMB_DIR.mkdir(exist_ok=True)

# Load existing metadata (if any) to preserve tags / favorites across runs
if META_FILE.exists():
    with open(META_FILE) as f:
        meta = json.load(f)
    print(f'[scan] Loaded existing metadata: {len(meta.get("assets", {}))} assets')
else:
    meta = {'version': 1, 'library_dir': str(lib_path),
            'created': datetime.now(timezone.utc).isoformat(),
            'assets': {}}
    print('[scan] No existing metadata; starting fresh')

# ── Format recognition ─────────────────────────────────────────────
# Map (extension, optional header magic) to format category
# Format rules. Order matters for .ply: 3DGS detection (header magic) runs
# FIRST in classify(), so the FORMAT_RULES entry for .ply is a fallback only.
# Suffix-only rules (SOG, SPZ, KSPLAT, etc.) don't need header inspection.
# .glb can be either a regular mesh OR a splat-bearing GLB with the
# KHR_gaussian_splatting extension. classify() detects this by file header.
FORMAT_RULES = [
    ('mesh_ply',  '.ply',  'MESH'),        # 3DGS gets caught above by header magic
    ('3dgs_sog',  '.sog',  '3DGS_COMPRESSED'),  # PlayCanvas native
    ('3dgs_spz',  '.spz',  '3DGS_COMPRESSED'),  # Niantic (mobile AR)
    ('3dgs_ksplat', '.ksplat', '3DGS_COMPRESSED'),  # mkkellogg
    ('3dgs_lcc',  '.lcc',  '3DGS_COMPRESSED'),  # XGRIDS
    ('mesh_glb',  '.glb',  'MESH'),        # splat-GLB gets caught above by header magic
    ('collision_glb', '-collision.glb', 'COLLISION_MESH'),  # voxel collision mesh
    ('3dgs_lod_meta', '-lod-meta.json', 'LOD_META'),  # PlayCanvas streamed SOG meta
    ('mesh_obj',  '.obj',  'MESH'),
    ('mesh_fbx',  '.fbx',  'MESH'),
    ('mesh_stl',  '.stl',  'MESH'),
    ('mesh_3mf',  '.3mf',  'MESH'),
    ('image',     '.png',  'IMAGE'),
    ('image',     '.jpg',  'IMAGE'),
    ('image',     '.jpeg', 'IMAGE'),
    ('image',     '.webp', 'IMAGE'),
    ('text',      '.txt',  'TEXT'),
    ('json',      '.json', 'META'),
    ('archive',   '.zip',  'ARCHIVE'),
]

def classify(path: pathlib.Path) -> str:
    """Classify a file by extension + (for .ply / .glb) header magic.

    Categories:
      - 3DGS_RAW         : raw TripoSplat .ply (3DGS Gaussian attributes)
      - 3DGS_PACKED     : .splat (32-byte packed, Antimatter15 format)
      - 3DGS_COMPRESSED : .sog / .spz / .ksplat / .lcc / .glb w/ KHR_gaussian_splatting
      - MESH             : regular triangle mesh (.glb / .gltf / .obj / .fbx / .ply / .stl / .3mf)
      - COLLISION_MESH   : voxel collision mesh (e.g. hero_collision.glb)
      - LOD_META         : PlayCanvas streamed-SOG lod-meta.json sidecar
      - IMAGE            : PNG / JPG / WEBP
      - TEXT / META / ARCHIVE / OTHER
    """
    name = path.name
    ext = path.suffix.lower()
    # Suffix-based rules that need a filename pattern (suffix isn't enough)
    if name.endswith('-collision.glb'):
        return 'COLLISION_MESH'
    if name == 'lod-meta.json':
        return 'LOD_META'
    if ext == '.ply':
        try:
            with open(path, 'rb') as f:
                header = f.read(4096).decode('latin-1', errors='ignore')
            if 'f_dc_' in header and 'opacity' in header and 'scale_0' in header:
                return '3DGS_RAW'
            elif 'face' in header or 'vertex_index' in header:
                return 'MESH'
            else:
                return 'PLY_UNKNOWN'
        except Exception:
            return 'PLY_UNKNOWN'
    if ext == '.glb':
        # glTF 2.0 binary. Could be a regular mesh OR a 3DGS scene that uses
        # the KHR_gaussian_splatting extension. Inspect the JSON header to tell.
        try:
            with open(path, 'rb') as f:
                head = f.read(8192).decode('latin-1', errors='ignore')
            if 'KHR_gaussian_splatting' in head:
                return '3DGS_COMPRESSED'
            else:
                return 'MESH'
        except Exception:
            return 'MESH'
    for fmt, fmt_ext, category in FORMAT_RULES:
        if ext == fmt_ext:
            return category
    return 'OTHER'

# ── Scan the library ────────────────────────────────────────────────
t0 = time.time()
if RECURSIVE:
    all_files = [p for p in lib_path.rglob(INCLUDE_GLOBS) if p.is_file()]
else:
    all_files = [p for p in lib_path.glob(INCLUDE_GLOBS) if p.is_file()]
if MAX_ASSETS:
    all_files = all_files[:MAX_ASSETS]
print(f'[scan] Found {len(all_files)} files in {time.time()-t0:.1f}s')

# Classify + extract metadata
t0 = time.time()
n_added = n_updated = 0
for p in all_files:
    rel = str(p.relative_to(lib_path))
    if rel in meta['assets']:
        # Preserve existing user metadata, just refresh file stats
        m = meta['assets'][rel]
        m['size_bytes'] = p.stat().st_size
        m['mtime'] = p.stat().st_mtime
        n_updated += 1
    else:
        m = {
            'path': rel,
            'filename': p.name,
            'format': classify(p),
            'size_bytes': p.stat().st_size,
            'mtime': p.stat().st_mtime,
            'tags': [],
            'favorite': False,
            'notes': '',
            'added': datetime.now(timezone.utc).isoformat(),
        }
        meta['assets'][rel] = m
        n_added += 1
print(f'[scan] Classified in {time.time()-t0:.1f}s: {n_added} new, {n_updated} updated')

# Format breakdown
from collections import Counter
fmt_counts = Counter(m['format'] for m in meta['assets'].values())
print('\n[scan] Format breakdown:')
for fmt, n in fmt_counts.most_common():
    print(f'  {fmt:15s}: {n:4d} files')

total_size = sum(m['size_bytes'] for m in meta['assets'].values())
print(f'\n[scan] Total: {len(meta["assets"])} assets, {total_size/1024/1024:.1f} MB')

# Save metadata
with open(META_FILE, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'\n[scan] Metadata saved to {META_FILE}')
print(f'[scan] Thumbnail dir: {THUMB_DIR}')
print(f'[scan] Move on to STEP 3 (Gradio UI).')


In [ ]:
# @title STEP 3 — Gradio UI: Browse, Tag, Preview, Favorite
# @markdown The main browser. Gallery with thumbnails, filter sidebar,
# @markdown click-to-preview (3D mesh in `<model-viewer>`, 3DGS gets file info
# @markdown + 'convert to mesh' button), tag editor, favorite toggle. All state
# @markdown persists in the metadata sidecar JSON from STEP 2.

import os, sys, json, time, pathlib, shutil, traceback
import gradio as gr
from IPython.display import clear_output, display, HTML
import numpy as np
from PIL import Image
import trimesh
import open3d as o3d

LIBRARY_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out')
META_FILE = LIBRARY_DIR / '_library_meta.json'
THUMB_DIR = LIBRARY_DIR / '_thumbs'

if not META_FILE.exists():
    raise SystemExit(f'[ui] Run STEP 2 first ({META_FILE} not found).')
with open(META_FILE) as f:
    META = json.load(f)
print(f'[ui] Loaded metadata: {len(META["assets"])} assets')

# ── Helpers ───────────────────────────────────────────────────────
def _save_meta():
    META['updated'] = __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat()
    with open(META_FILE, 'w') as f:
        json.dump(META, f, indent=2)

def _all_tags():
    """Return the set of all tags used across all assets."""
    s = set()
    for m in META['assets'].values():
        for t in m.get('tags', []):
            s.add(t)
    return sorted(s)

def _all_formats():
    return sorted({m['format'] for m in META['assets'].values()})

def _list_assets(tag_filter=None, format_filter=None, favorite_only=False, search=''):
    """Return list of (rel_path, thumbnail_or_None, format, size_mb) tuples
    matching the filters, sorted by path."""
    out = []
    for rel, m in META['assets'].items():
        if tag_filter and tag_filter not in m.get('tags', []):
            continue
        if format_filter and format_filter != m['format']:
            continue
        if favorite_only and not m.get('favorite', False):
            continue
        if search and search.lower() not in rel.lower():
            continue
        thumb = THUMB_DIR / (rel.replace('/', '__') + '.png')
        thumb_str = str(thumb) if thumb.exists() else None
        out.append((rel, thumb_str, m['format'], m['size_bytes'] / 1024 / 1024))
    out.sort(key=lambda x: x[0])
    return out

def _preview_html(rel_path):
    """Build an HTML preview block. Mesh files: inline `<model-viewer>`.
    3DGS files: metadata card with note that 3DGS needs an external viewer.
    Images: inline `<img>`. Other: file info."""
    if rel_path is None:
        return '<em>Select an asset from the gallery</em>'
    m = META['assets'].get(rel_path, {})
    if not m:
        return f'<em>Unknown asset: {rel_path}</em>'
    full_path = LIBRARY_DIR / rel_path
    fmt = m.get('format', 'OTHER')
    size_mb = m.get('size_bytes', 0) / 1024 / 1024
    tags = ', '.join(m.get('tags', [])) or '(none)'
    favorite = 'YES' if m.get('favorite', False) else 'no'
    notes = m.get('notes', '') or '(no notes)'
    if fmt in ('MESH', 'COLLISION_MESH') and full_path.suffix.lower() in ('.glb', '.gltf', '.obj', '.fbx', '.ply', '.stl', '.3mf'):
        return f'''
            <div style="font-family: sans-serif; max-width: 800px;">
              <h3>{rel_path}</h3>
              <model-viewer src="{full_path}" alt="{rel_path}" auto-rotate camera-controls style="width: 100%; height: 500px; background: #1a1a1a;"></model-viewer>
              <script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.5.0/model-viewer.min.js"></script>
              <p><b>Format:</b> {fmt} ({full_path.suffix}) | <b>Size:</b> {size_mb:.2f} MB<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
            </div>
        '''
    elif fmt == 'IMAGE':
        return f'''
            <div style="font-family: sans-serif; max-width: 800px;">
              <h3>{rel_path}</h3>
              <img src="{full_path}" style="max-width: 100%; max-height: 500px;">
              <p><b>Format:</b> {fmt} ({full_path.suffix}) | <b>Size:</b> {size_mb:.2f} MB<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
            </div>
        '''
    elif fmt in ('3DGS_RAW', '3DGS_PACKED', '3DGS_COMPRESSED'):
        # Per-format viewer links. SOG/GLB+KHR have direct PlayCanvas support;
        # others go to the community splat viewers.
        if fmt == '3DGS_COMPRESSED':
            ext = full_path.suffix.lower()
            if ext == '.sog':
                viewer_links = '<a href="https://playcanvas.com/viewer/" target="_blank">PlayCanvas Viewer</a> (native SOG support), <a href="https://antimatter15.com/splat/" target="_blank">Antimatter15</a>, <a href="https://gsplat.tech/" target="_blank">gsplat.tech</a>'
                viewer_note = 'SOG is PlayCanvas-native (~10x compressed). Drop into [playcanvas.com/viewer](https://playcanvas.com/viewer/) for instant preview.'
            elif ext == '.spz':
                viewer_links = '<a href="https://scaniverse.com/" target="_blank">Niantic Scaniverse</a> (mobile), <a href="https://antimatter15.com/splat/" target="_blank">Antimatter15</a> (SPZ support), <a href="https://gsplat.tech/" target="_blank">gsplat.tech</a>'
                viewer_note = 'SPZ is the smallest cross-platform splat format (~5% of original PLY). Mobile-friendly via Scaniverse.'
            elif ext == '.ksplat':
                viewer_links = '<a href="https://antimatter15.com/splat/" target="_blank">Antimatter15</a>, <a href="https://gsplat.tech/" target="_blank">gsplat.tech</a>'
                viewer_note = 'KSPLAT (mkkellogg format). Web-friendly, smaller than PLY.'
            elif ext == '.lcc':
                viewer_links = '<a href="https://lumalabs.ai/" target="_blank">LumaAI</a> (native LCC), <a href="https://antimatter15.com/splat/" target="_blank">Antimatter15</a>'
                viewer_note = 'LCC (XGRIDS Luma format). LumaAI viewer has native support.'
            else:
                # GLB + KHR_gaussian_splatting
                viewer_links = '<a href="https://playcanvas.com/viewer/" target="_blank">PlayCanvas Viewer</a> (KHR_gaussian_splatting), <a href="https://antimatter15.com/splat/" target="_blank">Antimatter15</a>, <a href="https://gsplat.tech/" target="_blank">gsplat.tech</a> (Three.js plugin)'
                viewer_note = 'GLB with the new <code>KHR_gaussian_splatting</code> glTF extension. Future-proof standard — every engine will eventually support it.'
        else:
            # RAW PLY or PACKED SPLAT
            viewer_links = '<a href="https://antimatter15.com/splat/" target="_blank">Antimatter15</a>, <a href="https://gsplat.tech/" target="_blank">gsplat.tech</a>, LumaAI'
            viewer_note = 'Raw 3DGS — 10x larger than compressed formats. Run <code>SplatTransform_Colab</code> Step 3 to convert to SOG/SPZ/GLB for game use.'
        return f'''
            <div style="font-family: sans-serif; max-width: 800px; padding: 20px; background: #fff8e1; border-left: 4px solid #ff9800;">
              <h3>{rel_path}</h3>
              <p><b>3DGS Gaussian Splat ({fmt})</b> — cannot be displayed in <code>&lt;model-viewer&gt;</code>.
              Use a 3DGS viewer: {viewer_links}.</p>
              <p>{viewer_note}</p>
              <p><b>For textured visual mesh from this 3DGS:</b> use <a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/SuGaR_Colab.ipynb" target="_blank">SuGaR_Colab</a> (sharp, 2-3 hrs) or <a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/GauStudio_Colab.ipynb" target="_blank">GauStudio_Colab</a> (smooth, ~10 min).</p>
              <p><b>Format:</b> {fmt} ({full_path.suffix}) | <b>Size:</b> {size_mb:.2f} MB<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
            </div>
        '''
    elif fmt == 'LOD_META':
        try:
            content = full_path.read_text()
            pretty = content[:2000]  # truncate for display
        except Exception as e:
            return f'<em>Failed to read {rel_path}: {e}</em>'
        return f'''
            <div style="font-family: sans-serif; max-width: 800px; padding: 20px; background: #e3f2fd; border-left: 4px solid #1976d2;">
              <h3>{rel_path}</h3>
              <p><b>PlayCanvas LOD metadata</b> — sidecar for streamed SOG loading. Drop the parent folder
              (containing the .sog files + this <code>lod-meta.json</code>) into the
              <a href="https://playcanvas.com/viewer/" target="_blank">PlayCanvas Viewer</a> for progressive loading.</p>
              <p><b>Format:</b> {fmt} ({full_path.suffix}) | <b>Size:</b> {size_mb:.2f} MB<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
              <details><summary><b>Show lod-meta.json contents</b></summary>
              <pre style="background: #1a1a1a; color: #e0e0e0; padding: 12px; border-radius: 4px; max-height: 300px; overflow: auto;">{pretty}</pre>
              </details>
            </div>
        '''
    else:
        return f'''
            <div style="font-family: sans-serif; max-width: 800px;">
              <h3>{rel_path}</h3>
              <p>No inline preview for <code>{fmt}</code> ({full_path.suffix}).</p>
              <p><b>Size:</b> {size_mb:.2f} MB<br>
              <b>Tags:</b> {tags} | <b>Favorite:</b> {favorite}<br>
              <b>Notes:</b> {notes}</p>
            </div>
        '''

def _build_gallery():
    return _list_assets(tag_filter=gr_state.get('tag'),
                        format_filter=gr_state.get('format'),
                        favorite_only=gr_state.get('fav'),
                        search=gr_state.get('search', ''))

gr_state = {'tag': None, 'format': None, 'fav': False, 'search': ''}

# ── Gradio UI ─────────────────────────────────────────────────
with gr.Blocks(title='AEI Asset Library Browser', theme=gr.themes.Soft(),
               css='footer {visibility: hidden}') as demo:
    gr.Markdown('# AEI Asset Library Browser\n'
                f'Library: `{LIBRARY_DIR}` — {len(META["assets"])} assets')
    with gr.Row():
        with gr.Column(scale=1, min_width=280):
            gr.Markdown('### Filter')
            search_box = gr.Textbox(value='', label='Search filename',
                                     placeholder='e.g. hero, glb, knight...', info='Filter by filename substring.')
            tag_dd = gr.Dropdown(choices=[None] + _all_tags(), value=None,
                                  label='Filter by tag', info='Show only assets with this tag.')
            fmt_dd = gr.Dropdown(choices=[None] + _all_formats(), value=None,
                                  label='Filter by format', info='MESH, 3DGS_RAW, IMAGE, etc.')
            fav_cb = gr.Checkbox(value=False, label='Favorites only', info='Show only assets you starred with the Favorite button.')
            gr.Markdown('### Edit selected asset')
            sel_path = gr.Textbox(value='', label='Selected path',
                                   interactive=False, info='Auto-populated on gallery click.')
            tags_box = gr.Textbox(value='', label='Tags (comma-separated)',
                                     placeholder='hero, vehicle, low-poly',
                                     info='Comma-separated. Used by Filter + exports.')
            notes_box = gr.Textbox(value='', label='Notes',
                                      placeholder='description, source, etc.',
                                      lines=3, info='Shown in preview + HTML portfolio.')
            fav_btn = gr.Button('Toggle favorite')
            save_btn = gr.Button('Save changes', variant='primary')
        with gr.Column(scale=3):
            gallery = gr.Gallery(value=_build_gallery(), columns=4, height=600,
                                 label='Assets (click to preview)',
                                 allow_preview=False)
            preview = gr.HTML(value=_preview_html(None), label='Preview')

    def on_select(evt: gr.SelectData):
        # Gallery uses (image, caption) tuples; evt.value is the chosen image path
        # which for thumbnails is the .png file. To map back to the asset
        # path, reverse the THUMB_DIR naming convention.
        chosen = evt.value
        if chosen is None:
            return '', '', '', _preview_html(None)
        thumb_path = pathlib.Path(chosen)
        rel = thumb_path.stem.replace('__', '/')
        m = META['assets'].get(rel, {})
        if not m:
            return rel, '', '', _preview_html(rel)
        return rel, ', '.join(m.get('tags', [])), m.get('notes', ''), _preview_html(rel)

    def on_search(s, t, f, fav):
        gr_state['search'] = s
        gr_state['tag'] = t
        gr_state['format'] = f
        gr_state['fav'] = fav
        return _list_assets(tag_filter=t, format_filter=f,
                            favorite_only=fav, search=s)

    def on_save(rel, tags_str, notes):
        if not rel or rel not in META['assets']:
            return gr.update(), gr.update(value='(nothing to save)')
        m = META['assets'][rel]
        m['tags'] = [t.strip() for t in tags_str.split(',') if t.strip()]
        m['notes'] = notes
        _save_meta()
        return _list_assets(tag_filter=gr_state.get('tag'),
                            format_filter=gr_state.get('format'),
                            favorite_only=gr_state.get('fav'),
                            search=gr_state.get('search', '')), \
               gr.update(value=f'Saved {rel} (tags={m["tags"]})')

    def on_fav(rel):
        if not rel or rel not in META['assets']:
            return gr.update(), gr.update(value='(select an asset first)')
        m = META['assets'][rel]
        m['favorite'] = not m.get('favorite', False)
        _save_meta()
        return _list_assets(tag_filter=gr_state.get('tag'),
                            format_filter=gr_state.get('format'),
                            favorite_only=gr_state.get('fav'),
                            search=gr_state.get('search', '')), \
               gr.update(value=f'{"FAVORITED" if m["favorite"] else "unfavorited"} {rel}')

    # Wire up
    search_box.change(on_search, [search_box, tag_dd, fmt_dd, fav_cb], gallery)
    tag_dd.change(on_search, [search_box, tag_dd, fmt_dd, fav_cb], gallery)
    fmt_dd.change(on_search, [search_box, tag_dd, fmt_dd, fav_cb], gallery)
    fav_cb.change(on_search, [search_box, tag_dd, fmt_dd, fav_cb], gallery)
    gallery.select(on_select, None, [sel_path, tags_box, notes_box, preview])
    save_btn.click(on_save, [sel_path, tags_box, notes_box], [gallery, preview])
    fav_btn.click(on_fav, sel_path, [gallery, preview])

def _welcome():
        n = len(META['assets'])
        return ('<div style="font-family: sans-serif; padding: 16px; background: #e3f2fd; border-radius: 6px;">'
                '<b>Welcome to the Asset Library Browser</b><br>'
                f'Library: <code>{LIBRARY_DIR}</code> — {n} assets.<br>'
                'Click any asset in the gallery to preview. Use the filter sidebar to narrow by tag, format, or favorites.<br>'
                'Run STEP 4 first to render thumbnails for mesh files.<br>'
                'Recognized formats: PLY/SPLAT/SOG/SPZ/KSPLAT/LCC/GLB+KHR_gaussian_splatting (3DGS), '
                'GLB/OBJ/FBX/PLY/STL/3MF (mesh), -collision.glb (voxel collision), '
                'lod-meta.json (PlayCanvas streamed SOG), PNG/JPG/WEBP, JSON, ZIP, TXT.</div>')
demo.load(_welcome, None, preview)

clear_output()
demo.queue(max_size=4, default_concurrency_limit=2)
demo.launch(share=False, show_error=True, height=900, quiet=False)


In [ ]:
# @title STEP 4 — Batch Render Thumbnails (optional)
# @markdown Renders 256×256 PNG thumbnails for every MESH asset. The Gradio
# @markdown gallery shows these alongside filenames. 3DGS assets are skipped
# @markdown (no easy in-notebook renderer; the Gradio UI shows a file icon).
# @markdown Run once after the library grows; idempotent (only renders missing).

import os, sys, json, time, pathlib
import numpy as np
from PIL import Image
import trimesh
from tqdm import tqdm

LIBRARY_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out')
META_FILE = LIBRARY_DIR / '_library_meta.json'
THUMB_DIR = LIBRARY_DIR / '_thumbs'
THUMB_DIR.mkdir(exist_ok=True)
with open(META_FILE) as f:
    META = json.load(f)

THUMB_SIZE = 256  # @param {type:"slider", min:64, max:1024, step:64}
# @markdown *Thumbnail edge size in pixels. 256 = gallery-friendly. 512 = preview-quality.*
RENDER_RES = 256  # @param {type:"slider", min:128, max:2048, step:64}
# @markdown *Mesh render resolution. Higher = better quality, slower.*
OVERWRITE = False  # @param {type:"boolean"}
# @markdown *Re-render existing thumbnails. Default False (only render missing).*
MESH_FORMATS = ('MESH',)  # only render mesh files; skip 3DGS / images / text

def _render_mesh_thumb(mesh_path, out_path, size=256, res=256):
    """Render a trimesh mesh to a PNG thumbnail via offscreen pyrender.
    Falls back to a placeholder PNG if pyrender/OpenGL is unavailable."""
    try:
        m = trimesh.load(str(mesh_path), force='mesh')
        if not isinstance(m, trimesh.Trimesh) or len(m.vertices) == 0:
            return False
        # Center + normalize to unit cube for consistent thumbnail framing
        m.vertices -= (m.vertices.min(axis=0) + m.vertices.max(axis=0)) / 2.0
        extent = float((m.vertices.max(axis=0) - m.vertices.min(axis=0)).max())
        if extent > 1e-9:
            m.vertices /= extent
        # Use trimesh's built-in offscreen renderer
        try:
            scene = m.scene()
            png = scene.save_image(resolution=(res, res), background=[20, 20, 30, 255])
            if png:
                with open(out_path, 'wb') as f:
                    f.write(png)
                return True
        except Exception as e:
            pass
        # Fallback: try pyglet
        try:
            from trimesh.viewer import SceneViewer
            return False  # not a real fallback, just signal failure
        except Exception:
            pass
        return False
    except Exception as e:
        print(f'[thumb] render failed for {mesh_path}: {e}')
        return False

def _placeholder_thumb(out_path, size=256, label=''):
    """Generate a placeholder PNG with the file extension as a label."""
    img = Image.new('RGB', (size, size), (40, 40, 50))
    from PIL import ImageDraw, ImageFont
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 32)
        small = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 18)
    except Exception:
        font = ImageFont.load_default()
        small = font
    if label:
        # Center the label
        bbox = draw.textbbox((0, 0), label, font=font)
        w = bbox[2] - bbox[0]
        draw.text(((size - w) / 2, size / 2 - 20), label, fill=(220, 220, 230), font=font)
    img.save(out_path)

t0 = time.time()
n_total = n_rendered = n_skipped = n_failed = 0
for rel, m in tqdm(META['assets'].items(), desc='thumbnails'):
    if m['format'] not in MESH_FORMATS:
        n_skipped += 1
        continue
    n_total += 1
    thumb_path = THUMB_DIR / (rel.replace('/', '__') + '.png')
    if thumb_path.exists() and not OVERWRITE:
        continue
    src_path = LIBRARY_DIR / rel
    if _render_mesh_thumb(src_path, thumb_path, size=THUMB_SIZE, res=RENDER_RES):
        n_rendered += 1
    else:
        # Placeholder with the extension as label (e.g. "GLB")
        ext = src_path.suffix.lstrip('.').upper()
        _placeholder_thumb(thumb_path, size=THUMB_SIZE, label=ext)
        n_failed += 1
print(f'\n[thumb] Done in {time.time()-t0:.1f}s')
print(f'  MESH assets: {n_total}, rendered: {n_rendered}, failed (placeholder): {n_failed}, skipped (non-mesh): {n_skipped}')
print(f'\n[Done] Thumbnails in {THUMB_DIR}/. Refresh the Gradio UI in STEP 3 to see them.')


In [ ]:
# @title STEP 5 — Export: Unity / Godot / Static HTML Portfolio / CSV
# @markdown Exports the library in formats game engines and portfolio sites
# @markdown can ingest. Each export respects the current filter (tag / format /
# @markdown favorite / search) so you can ship a curated subset.

import os, json, time, shutil, zipfile, pathlib
import pandas as pd
from datetime import datetime, timezone

LIBRARY_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out')
META_FILE = LIBRARY_DIR / '_library_meta.json'
THUMB_DIR = LIBRARY_DIR / '_thumbs'
EXPORT_DIR = LIBRARY_DIR / '_exports'
EXPORT_DIR.mkdir(exist_ok=True)
with open(META_FILE) as f:
    META = json.load(f)

EXPORT_FORMATS = ['unity', 'godot', 'html_portfolio', 'csv_manifest']  # @param ["unity", "godot", "html_portfolio", "csv_manifest"]
# @markdown *Comma-separated list. Each export goes to /content/drive/MyDrive/AEI_3D_Out/_exports/<name>/.*
INCLUDED_FORMATS = ['MESH']  # @param ["MESH", "3DGS_RAW", "3DGS_PACKED", "IMAGE", "ALL"]
# @markdown *Filter to specific asset formats. MESH only (skip 3DGS). Use ALL to include everything.*
INCLUDED_TAGS = ''  # @param {type:"string"}
# @markdown *Comma-separated tag whitelist. Empty = no filter (include all). Example: 'hero, vehicle'.*
FAVORITES_ONLY = False  # @param {type:"boolean"}
# @markdown *Only export favorited assets.*

def _filter_assets():
    """Return the asset paths matching INCLUDED_FORMATS, INCLUDED_TAGS, FAVORITES_ONLY."""
    fmt_set = set(INCLUDED_FORMATS) if INCLUDED_FORMATS != ['ALL'] else None
    if INCLUDED_TAGS.strip():
        tag_set = set(t.strip() for t in INCLUDED_TAGS.split(',') if t.strip())
    else:
        tag_set = None
    out = []
    for rel, m in META['assets'].items():
        if fmt_set and m['format'] not in fmt_set:
            continue
        if tag_set and not (set(m.get('tags', [])) & tag_set):
            continue
        if FAVORITES_ONLY and not m.get('favorite', False):
            continue
        out.append((rel, m))
    return sorted(out)

assets = _filter_assets()
print(f'[export] Filtered: {len(assets)} assets match criteria')
if not assets:
    print('[export] WARNING: no assets match. Loosen filters.')
    print('[export]   Try: INCLUDED_FORMATS=[MESH,3DGS_RAW], FAVORITES_ONLY=False')
print()

ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')

if 'csv_manifest' in EXPORT_FORMATS:
    print('[export] CSV manifest ...')
    csv_path = EXPORT_DIR / f'manifest_{ts}.csv'
    rows = []
    for rel, m in assets:
        rows.append({
            'path': rel,
            'filename': m.get('filename'),
            'format': m.get('format'),
            'size_mb': round(m.get('size_bytes', 0) / 1024 / 1024, 3),
            'tags': ', '.join(m.get('tags', [])),
            'favorite': m.get('favorite', False),
            'notes': m.get('notes', ''),
            'added': m.get('added', ''),
            'mtime': datetime.fromtimestamp(m.get('mtime', 0), timezone.utc).isoformat() if m.get('mtime') else '',
        })
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f'  -> {csv_path} ({len(rows)} rows)')

if 'unity' in EXPORT_FORMATS:
    print('[export] Unity AssetBundle-like folder structure ...')
    unity_dir = EXPORT_DIR / f'unity_{ts}'
    unity_dir.mkdir(exist_ok=True)
    (unity_dir / 'Assets').mkdir(exist_ok=True)
    (unity_dir / 'Assets' / 'AEI_Library').mkdir(exist_ok=True)
    (unity_dir / 'Assets' / 'AEI_Library' / 'Meshes').mkdir(exist_ok=True)
    (unity_dir / 'Assets' / 'AEI_Library' / 'Thumbs').mkdir(exist_ok=True)
    n = 0
    for rel, m in assets:
        if m['format'] != 'MESH':
            continue
        src = LIBRARY_DIR / rel
        dst = unity_dir / 'Assets' / 'AEI_Library' / 'Meshes' / m['filename']
        if not dst.exists():
            shutil.copy2(src, dst)
        thumb = THUMB_DIR / (rel.replace('/', '__') + '.png')
        if thumb.exists():
            shutil.copy2(thumb, unity_dir / 'Assets' / 'AEI_Library' / 'Thumbs' / thumb.name)
        n += 1
    # README with import instructions
    readme = unity_dir / 'Assets' / 'AEI_Library' / 'README.md'
    readme.write_text(f'''# AEI Asset Library \u2192 Unity

**Generated:** {datetime.now(timezone.utc).isoformat()}
**Source:** {LIBRARY_DIR}
**Asset count:** {n} meshes (Unity prefers .fbx for rigged assets, but .glb and .obj import fine for static props)

## Import into Unity

1. Copy the `Assets/AEI_Library` folder into your Unity project's `Assets/`.
2. Unity auto-imports .glb / .obj / .fbx. The default scale is 1 unit = 1 meter
   (matches our game-asset transform defaults).
3. For the thumbs folder: select any mesh in the Project window and the
   thumbnail preview will appear.
4. To bundle into an AssetBundle: select all meshes in `AEI_Library/Meshes/`,
   set their AssetBundle name in the Inspector (e.g. `aei/library`),
   then build via `BuildPipeline.BuildAssetBundles`.

## Notes

- These meshes are untextured. Texture them in your DCC tool of choice
  (Blender, Substance Painter) before shipping.
- Use Unity's mesh decimation (in the Model import settings, Model tab)
  to make per-platform LODs.
''')
    print(f'  -> {unity_dir}/ (with README, {n} meshes copied)')

if 'godot' in EXPORT_FORMATS:
    print('[export] Godot mesh library (file copy + .import sidecar) ...')
    godot_dir = EXPORT_DIR / ('godot_' + ts)
    godot_dir.mkdir(exist_ok=True)
    n = 0
    for rel, m in assets:
        if m['format'] != 'MESH':
            continue
        src = LIBRARY_DIR / rel
        # Godot auto-detects .glb/.obj/.fbx/.stl on import.
        # Just copy the file; Godot will generate the .import sidecar on first scan.
        target = godot_dir / m['filename']
        if not target.exists():
            shutil.copy2(src, target)
        n += 1
    # README with import instructions
    readme = godot_dir / 'README.md'
    readme_lines = [
        '# AEI Asset Library to Godot',
        '',
        'Drop this entire folder into your Godot project root, then in the Godot editor:',
        '- FileSystem panel: right-click to Reimport',
        '- For .glb/.gltf: enable the import-on-drop option',
        '- For .fbx: install the FBX importer plugin',
        '',
        'Meshes are untextured. Texture them in Blender before shipping.',
    ]
    readme.write_text('\n'.join(readme_lines))
    print('  -> {godot_dir}/ ({n} mesh files + README)'.format(godot_dir=str(godot_dir), n=n))
    print('  Drop the folder into your Godot project root.')


print('\n[Done] Exports in /content/drive/MyDrive/AEI_3D_Out/_exports/.')


In [ ]:
# @title STEP 6 — Stats Dashboard
# @markdown A textual summary of your library: format breakdown, top tags,
# @markdown biggest files, missing/broken files report. Useful for QA before
# @markdown shipping a 200+ asset pack.

import os, json, pathlib
from collections import Counter
import pandas as pd

LIBRARY_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out')
META_FILE = LIBRARY_DIR / '_library_meta.json'
with open(META_FILE) as f:
    META = json.load(f)
assets = META['assets']
n = len(assets)

print('=== AEI Asset Library Stats ===\n')
print(f'Library dir: {LIBRARY_DIR}')
print(f'Total assets: {n}')
total_size = sum(m.get('size_bytes', 0) for m in assets.values())
print(f'Total size:   {total_size/1024/1024:.1f} MB ({total_size/1024**3:.2f} GB)\n')

# Format breakdown
print('--- Format breakdown ---')
fmt_counts = Counter(m['format'] for m in assets.values())
fmt_sizes = Counter()
for m in assets.values():
    fmt_sizes[m['format']] += m.get('size_bytes', 0)
for fmt, n_f in fmt_counts.most_common():
    sz_mb = fmt_sizes[fmt] / 1024 / 1024
    print(f'  {fmt:15s}: {n_f:4d} files, {sz_mb:7.1f} MB')

# Top tags
print('\n--- Top 15 tags ---')
tag_counts = Counter()
for m in assets.values():
    for t in m.get('tags', []):
        tag_counts[t] += 1
for tag, n_t in tag_counts.most_common(15):
    print(f'  {tag:30s}: {n_t:4d} assets')

# Favorites
n_fav = sum(1 for m in assets.values() if m.get('favorite', False))
print(f'\nFavorited: {n_fav} ({100*n_fav/max(1,n):.1f}%)')

# Biggest files
print('\n--- Top 10 biggest files ---')
biggest = sorted(assets.values(), key=lambda m: m.get('size_bytes', 0), reverse=True)[:10]
for m in biggest:
    sz = m.get('size_bytes', 0) / 1024 / 1024
    print(f'  {sz:7.1f} MB  {m["format"]:12s}  {m["path"]}')

# Missing/broken files
print('\n--- Missing files (in metadata but not on disk) ---')
missing = []
for rel, m in assets.items():
    p = LIBRARY_DIR / rel
    if not p.exists():
        missing.append(rel)
if missing:
    for rel in missing[:20]:
        print(f'  MISSING: {rel}')
    if len(missing) > 20:
        print(f'  ... and {len(missing) - 20} more')
else:
    print('  (none \u2014 all metadata entries have files on disk)')

# Orphaned files (on disk but not in metadata)
print('\n--- Orphaned files (on disk but not in metadata) ---')
all_disk = set()
for p in LIBRARY_DIR.rglob('*'):
    if p.is_file() and not p.name.startswith('.') and '_thumbs' not in p.parts and '_exports' not in p.parts and p.name != '_library_meta.json':
        all_disk.add(str(p.relative_to(LIBRARY_DIR)))
all_meta = set(assets.keys())
orphans = all_disk - all_meta
if orphans:
    for rel in sorted(orphans)[:20]:
        print(f'  ORPHAN: {rel}')
    if len(orphans) > 20:
        print(f'  ... and {len(orphans) - 20} more')
else:
    print('  (none)')
print('\n[Done] Run STEP 2 again to refresh metadata if you see MISSING/ORPHAN.')


In [ ]:
# @title STEP 7 — Keep-Alive (Gradio stays alive)
# @markdown Standard pattern. Gradio will keep running as long as this kernel is
# @markdown alive. The KEEP_ALIVE_JS prevents Colab from idling you out.

import IPython
from google.colab import output
KEEP_ALIVE_JS = """
function KeepAlive() {
    setInterval(() => {
        console.log("keep-alive:", new Date().toISOString());
        google.colab.kernel.proxyProxy(0).then(() => {}).catch(() => {});
    }, 30 * 60 * 1000);
}
KeepAlive();
"""
display(IPython.display.Javascript(KEEP_ALIVE_JS))
print('[KeepAlive] Started \u2014 pings every 30 min.')


In [ ]:
# @title STEP 8 — Help / Format Reference / Known Issues
# @markdown Reference material. Read this if anything in the browser went wrong.

help_md = '''
# Asset Library Browser \u2014 Help, Format Reference, and Known Issues

## What this notebook is for

After running TripoSplat + GauStudio / SuGaR + Mesh Optimizer on your 200+ images, you have a folder full of assets but no way to:
- Browse them all at once
- Preview 3D meshes in your browser
- Tag / favorite the good ones
- Export a curated subset to Unity / Godot / a portfolio site

This notebook is the missing piece.

## Format reference

| Format | Category | Preview support |
|--------|----------|------------------|
`.ply` (3DGS-extended) | `3DGS_RAW` | Metadata card + link to external 3DGS viewer |
`.splat` (packed) | `3DGS_PACKED` | Metadata card + link to Antimatter15 viewer |
`.glb` (binary glTF) | `MESH` | Inline `<model-viewer>` (best UX) |
`.obj` (Wavefront) | `MESH` | Inline `<model-viewer>` |
`.fbx` (FBX 7.4) | `MESH` | Inline `<model-viewer>` |
`.ply` (mesh) | `MESH` | Inline `<model-viewer>` |
`.stl` (3D print) | `MESH` | Inline `<model-viewer>` |
`.3mf` (3D Manuf.) | `MESH` | Inline `<model-viewer>` |
`.png` / `.jpg` / `.webp` | `IMAGE` | Inline `<img>` |
`.txt` / `.json` / `.zip` | `TEXT` / `META` / `ARCHIVE` | File info only |

## 3DGS preview workaround

`<model-viewer>` doesn't support 3DGS Gaussian Splats. For 3DGS files you get a metadata card with a link to Antimatter15's online viewer (or LumaAI, gsplat.tech). To get inline previews, run GauStudio on the 3DGS PLY \u2192 get a GLB mesh \u2192 come back to the browser and preview the GLB.

## Thumbnails

STEP 4 renders 256\u00d7256 PNG thumbnails via trimesh's offscreen renderer for every MESH asset. 3DGS files get a placeholder with the extension as a label. Re-run after adding new assets.

## Exports

| Export | What's in it | Game-engine import |
|--------|--------------|-------------------|
Unity | `Assets/AEI_Library/Meshes/` + thumbnails + README | Copy `Assets/AEI_Library/` into your Unity project's `Assets/` |
Godot | `*.tres` mesh resources + actual mesh files | Copy into your Godot project's root, then use `FileSystem > Import` |
HTML portfolio | Self-contained `index.html` + thumbs + meshes | Upload to GitHub Pages, Netlify, or any static host |
CSV manifest | `manifest_<ts>.csv` with path, format, tags, size, etc. | Open in Excel / Google Sheets for inventory |

## Known issues

1. **3DGS files get no thumbnail**: no easy in-notebook 3DGS renderer. Workaround: run GauStudio on them first, then re-scan.
2. **Large meshes (>50 MB) slow to render thumbnails**: trimesh's offscreen renderer loads the whole mesh in memory. For 200 assets, expect 2-5 min total render time.
3. **`<model-viewer>` doesn't show texture for GLB if it references external files**: our GLBs are self-contained (texture embedded) so this shouldn't happen. If you see a black mesh, check that the GLB has a `.bin` or embedded texture.
4. **Gradio URL only works while the kernel is alive**: when the session ends, the Gradio URL dies. To share, export the HTML portfolio (STEP 5) and host it on a static site.
5. **Drive sync conflicts**: if multiple notebooks are writing to the same `_library_meta.json`, you can lose tag changes. STEP 2 reads the sidecar on each run; STEP 3's save button writes immediately. Avoid running two browsers in parallel.

## SplatTransform output formats (from [SplatTransform_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/SplatTransform_Colab.ipynb))

The PlayCanvas `splat-transform` post-processor takes raw 3DGS `.ply` files and converts them to compressed game-engine formats. If you've run SplatTransform_Colab on the same library, you'll see those files here too. Key formats to recognize:

- **`.sog`** (PlayCanvas native, ~10x compression) — open in [playcanvas.com/viewer](https://playcanvas.com/viewer/)
- **`.spz`** (Niantic, smallest, mobile-friendly) — Scaniverse
- **`.ksplat`** (mkkellogg) — community 3DGS viewers
- **`.lcc`** (XGRIDS) — LumaAI native
- **`.glb` with `KHR_gaussian_splatting`** (the new glTF 2.0 standard extension) — auto-detected by header magic. Future-proof for Three.js / Babylon.js / Unity / Unreal.
- **`-collision.glb`** (voxel marching-cubes mesh, runtime physics) — auto-detected by filename suffix. NOT a textured visual mesh.
- **`lod-meta.json`** (PlayCanvas streamed SOG metadata) — auto-detected by exact filename. Pair with the .sog files in the same folder for progressive loading.

The browser recognizes all of these (via the format classification in STEP 2) and shows a metadata card with the right viewer link for each. Run STEP 4 to render thumbnails for the mesh files; 3DGS files get no thumbnail (no easy in-notebook renderer) but are still scannable + filterable.

## See also

- **TripoSplat_Colab.ipynb** \u2014 generate 3DGS + mesh from a single image
- **SuGaR_Colab.ipynb** \u2014 high-quality mesh from 3DGS PLY (sharp, slow)
- **GauStudio_Colab.ipynb** \u2014 fast mesh from 3DGS PLY (smooth, fast)
- **Mesh_Optimizer_Colab.ipynb** \u2014 post-process meshes (UV unwrap, fill holes, etc.)
- **TripoSplat STEP 7** \u2014 batch generation (use this to build the 200+ library)
- **TripoSplat STEP 8** \u2014 post-process existing meshes (use to convert 3DGS to mesh for previews)
'''
from IPython.display import Markdown, display
display(Markdown(help_md))
print(help_md)
